# Pymol Visualization
## Objective

The objective of this tutorial is to learn how to fully utilize `pymol_visualize()` method.

This workflow allows you to:

- Generate Python script for PyMOL visualization
- Visualize different regions using PyMOL scene function
- Create separate directories when making multiple visualization script

In this specific example, we are using ketosteroid isomerase (KSI) as the model system. The structure for KSI is obtained from the PDB [1OH0](https://doi.org/10.2210/pdb1OH0/pdb) and MM-minimized prior to this tutorial.

## Classes used in this example

- [GenerateModel](https://qmzyme.readthedocs.io/en/latest/API/QMzyme.GenerateModel.html)
- [QM_Method](https://qmzyme.readthedocs.io/en/latest/API/QMzyme.CalculateModel.html#qm-treatment)
- [CA_terminal TruncationScheme](https://qmzyme.readthedocs.io/en/latest/API/QMzyme.TruncationSchemes.html#QMzyme.TruncationSchemes.CA_terminal)
- [DistanceCutoff SelectionScheme](https://qmzyme.readthedocs.io/en/latest/API/QMzyme.SelectionSchemes.html#QMzyme.SelectionSchemes.DistanceCutoff)

## Required Files

To start, you will need:

- A fully prepped and protonated PDB

In [6]:
# Here are the necesary imports for this tutorial!

import QMzyme 
from QMzyme.data import PDB
from QMzyme.SelectionSchemes import DistanceCutoff

## Visualizing the Full Model

To start with, let's initialize QMzymeModel and simply visualize the loaded structure file. This can be done by initializing QMzymeModel using `GenerateModel()` and using `pymol_visualize()` on the model to get python script for visualization.

In [7]:
model = QMzyme.GenerateModel(PDB)
model.pymol_visualize()


Charge information not present. QMzyme will try to guess region charges based on residue names consistent with AMBER naming conventions (i.e., aspartate: ASP --> Charge: -1, aspartic acid: ASH --> Charge: 0.). See QMzyme.data.residue_charges for the full set.


Now, when using this, you might have realized that the structure itself is hidden in the front interface. However, when you navigate towards `SCENES` tab in PyMOL, you can see that we see our starting structure!

To organize things better, you can specify the output_folder. Using this will create a directory with the specified name, and will store the Python script and structure files needed to make the PyMOL visualization!

In [3]:
# This is done simply by adding an argument, output_folder="str"
model.pymol_visualize(output_dir="full model")

## Visualizing the QMzyme-prepared structure

In addition to viewing the full model, `pymol_visualize()` can be used to examine the construction history of the region. Let's create a simple workflow using QMzyme, and use `pymol_visualize()` to see the QM region used for input file generation!

In [ ]:
# We first initilize QMzymeModel by loading structure file.
# The initial pdb file should be prepared prior (hydrogens must be present).
model = QMzyme.GenerateModel(PDB)

# This adds unknown residue charge.
QMzyme.data.residue_charges.update({'EQU': -1})

# This is used to set catalytic center.
model.set_catalytic_center(selection='resid 263')

# Use DistanceCutoff to create 3 Å region around catalytic center.
model.set_region(selection=DistanceCutoff, cutoff=3)

# Selecting and freeznig CA atoms.
c_alpha_atoms = model.cutoff_3.get_atoms(attribute='name', value='CA')
model.cutoff_3.set_fixed_atoms(atoms=c_alpha_atoms)

# Set method to a region.
qm_method = QMzyme.QM_Method(
    basis_set='6-31G*', 
    functional='wB97X-D3', 
    qm_input='OPT FREQ', 
    program='orca'
)
qm_method.assign_to_region(region=model.cutoff_3)

# Truncate the model
model.truncate()

# Write QM input file
model.write_input()

# Create Python script for visualization
model.pymol_visualize(output_dir="cutoff_3")

## Visualizing the .pkl file

After running a full workflow, QMzyme will create a pickle file, which will contain the construction history of the QM input file. The pickle file can be loaded to recreate the QMzymeModel for subsequent structural analysis!

In [5]:
model = QMzyme.GenerateModel(pickle_file=f'1oh0.pkl')
model.pymol_visualize(output_dir="pickled_model")

## References Cited

The PyMOL Molecular Graphics System, Version 3.0 Schrödinger, LLC.